In [ ]:
!pip install torch esm pandas
!pip install fair-esm --upgrade
!pip install biopython

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.0/58.0 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 102.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 6.6 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import esm
import pandas as pd
from Bio import SeqIO

# === Load ESM-2 (33-layer, 650M parameter model) ===
print("📥 Loading ESM-2 650M (33 layers)...")
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
batch_converter = alphabet.get_batch_converter()
model.eval()  # evaluation mode

# === Load sequences from FASTA ===
fasta_file = "/content/drive/MyDrive/Colab Notebooks/test-negative.fasta"
sequences = [(record.id, str(record.seq)) for record in SeqIO.parse(fasta_file, "fasta")]
print(f"✅ Loaded {len(sequences)} sequences from FASTA")

# === Prepare input batches ===
batch_labels, batch_strs, batch_tokens = batch_converter(sequences)

# === Compute embeddings (Layer 33) ===
print("🧠 Extracting embeddings from layer 33...")
with torch.no_grad():
    results = model(batch_tokens, repr_layers=[33])
    token_representations = results["representations"][33]

# === Average token embeddings (excluding special tokens) ===
sequence_representations = []
for i, (_, seq) in enumerate(sequences):
    rep = token_representations[i, 1:len(seq)+1].mean(0)
    sequence_representations.append(rep)

# === Convert to DataFrame and save ===
sequence_representations = torch.stack(sequence_representations).numpy()
df = pd.DataFrame(sequence_representations, index=[s[0] for s in sequences])
output_file = "ESM2_650M_Layer33_Isuccs_TestN.csv"
df.to_csv(output_file)

print(f"✅ Features extracted from Layer 33 and saved as: {output_file}")


Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t6_8M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t6_8M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t6_8M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t6_8M_UR50D-contact-regression.pt
